### Explaining self-attention


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

### Manual Head


In [2]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32  # batch, time, channels
x = torch.randn(B, T, C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)  # (B, T, 16)
q = query(x)  # (B, T, 16)
wei = (
    q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
)  # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

v = value(x)  # (B, T, 16)
out_manual = wei @ v
# out_manual = wei @ x

out_manual.shape

torch.Size([4, 8, 16])

In [3]:
out_manual[0][0]

tensor([-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
        -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
       grad_fn=<SelectBackward0>)

### Class Head


In [4]:
block_size = T
n_embd = C


class Head(nn.Module):
    """one head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B, T, C = x.shape
        k = self.key(x)  # (B,T,hs)
        q = self.query(x)  # (B,T,hs)
        # compute attention scores ("affinities")
        wei = (
            q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        )  # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # (B, T, T)
        wei = F.softmax(wei, dim=-1)  # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x)  # (B,T,hs)
        out = wei @ v  # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out


In [5]:
torch.manual_seed(1337)
h = Head(head_size)
out_head = h(x)
out_head.shape

torch.Size([4, 8, 16])

In [6]:
out_head[0][0]

tensor([ 0.1196, -0.3013,  0.3629,  1.1771,  1.1385, -0.2554,  0.1454, -0.2944,
        -0.7020, -1.0308,  0.7436, -0.8098, -0.6669,  0.0912, -0.0061,  0.1983],
       grad_fn=<SelectBackward0>)

In [7]:
torch.allclose(out_manual[0][0], out_head[0][0], atol=1e-6)

False

### Copy manual weights to Head Class


In [8]:
h = Head(head_size)

with torch.no_grad():
    h.key.weight.copy_(key.weight)
    h.query.weight.copy_(query.weight)
    h.value.weight.copy_(value.weight)

out_head = h(x)
out_head.shape

torch.Size([4, 8, 16])

In [9]:
out_head[0][0]

tensor([-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
        -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
       grad_fn=<SelectBackward0>)

In [10]:
torch.allclose(out_manual[0][0], out_head[0][0], atol=1e-6)

True

if we consume random numbers same as x.shape, then layers weights would be initialize with same random numbers and we don't need to copy them to get "out" of same values


In [11]:
torch.manual_seed(1337)
_ = torch.randn(B, T, C)
h = Head(head_size)
out_head = h(x)
out_head.shape

torch.Size([4, 8, 16])

In [12]:
out_head[0][0]

tensor([-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
        -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
       grad_fn=<SelectBackward0>)

In [13]:
torch.allclose(out_manual[0][0], out_head[0][0], atol=1e-6)

True

### Multi-Head Self-Attention


In [14]:
class MultiHeadAttention(nn.Module):
    """multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return out

In [15]:
torch.manual_seed(1337)
n_head = 4
assert n_embd % n_head == 0
head_size = n_embd // n_head
mha = MultiHeadAttention(n_head, head_size)
out_multi = mha(x)
out_multi.shape

torch.Size([4, 8, 32])

In [16]:
out_multi[0][0]

tensor([-0.6390,  1.3943,  0.5319, -1.0466, -0.7767,  0.6645,  0.0788,  0.6486,
        -0.7020, -1.0308,  0.7436, -0.8098, -0.6669,  0.0912, -0.0061,  0.1983,
        -0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         0.0544,  0.0762,  0.2364, -0.1792, -0.2908,  0.3348, -0.9431, -0.1245],
       grad_fn=<SelectBackward0>)

### Causal Self-Attention


In [17]:
import math


class CausalSelfAttention(nn.Module):
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        # output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.n_head = n_head
        self.n_embd = n_embd
        # not really a 'bias', more of a mask, but following the OpenAI/HF naming though
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(block_size, block_size)).view(
                1, 1, block_size, block_size
            ),
        )

    def forward(self, x):
        B, T, C = (
            x.size()
        )  # batch size, sequence length, embedding dimensionality (n_embd)
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh is "number of heads", hs is "head size", and C (number of channels) = nh * hs
        # e.g. in GPT-2 (124M), n_head=12, hs=64, so nh*hs=C=768 channels in the Transformer
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(
            1, 2
        )  # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(
            1, 2
        )  # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // n_head).transpose(1, 2)  # (B, nh, T, hs)
        # attention (materializes the large (T,T) matrix for all the queries and keys)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v  # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = (
            y.transpose(1, 2).contiguous().view(B, T, C)
        )  # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y

In [18]:
torch.manual_seed(1337)
csa = CausalSelfAttention(n_head, n_embd)
out_causal = csa(x)
out_causal.shape

torch.Size([4, 8, 32])

In [19]:
out_causal[0][0]

tensor([-0.2865, -0.0934, -0.1899, -0.6170, -0.1939,  0.1832, -0.4707, -0.0295,
        -0.0381, -0.1773, -0.5618,  0.0888, -0.1116,  0.2478, -0.3394,  0.4344,
        -0.4360, -0.4969,  0.4462, -0.5161, -0.0011,  0.5054,  0.5378,  0.2709,
        -0.0904,  0.2337, -0.0723,  0.2000, -0.0108,  0.1890, -0.0371, -0.2161],
       grad_fn=<SelectBackward0>)

In [20]:
torch.allclose(out_multi[0][0], out_causal[0][0], atol=1e-6)

False

### Experiment to copy MHA weights into CSA to get same output


In [21]:
csa = CausalSelfAttention(n_head, n_embd)

with torch.no_grad():
    q_weight = torch.cat([h.query.weight for h in mha.heads], dim=0)
    k_weight = torch.cat([h.key.weight for h in mha.heads], dim=0)
    v_weight = torch.cat([h.value.weight for h in mha.heads], dim=0)
    csa.c_attn.weight.copy_(torch.cat([q_weight, k_weight, v_weight], dim=0))
    csa.c_attn.bias.zero_()
    csa.c_proj.weight.copy_(torch.eye(n_embd))
    csa.c_proj.bias.zero_()

out_causal = csa(x)
out_causal.shape

torch.Size([4, 8, 32])

In [22]:
out_causal[0][0]

tensor([-0.6390,  1.3943,  0.5319, -1.0466, -0.7767,  0.6645,  0.0788,  0.6486,
        -0.7020, -1.0308,  0.7436, -0.8098, -0.6669,  0.0912, -0.0061,  0.1983,
        -0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         0.0544,  0.0762,  0.2364, -0.1792, -0.2908,  0.3348, -0.9431, -0.1245],
       grad_fn=<SelectBackward0>)

In [23]:
torch.allclose(out_multi[0][0], out_causal[0][0], atol=1e-6)

True